# Preparação de Dados
---
Elaboração da camada _gold_, através da criação de tabelas fato e dimensão.  

A tabela "bps_2024_a_2026" consolidada na camada _silver_ é a fonte única de informação da camada _gold_. As dimensões são populadas a partir dela e, posteriormente, a tabela fato é carregada por meio do relacionamento entre os registros da _silver_ e as dimensões.  

Além disso, são criadas duas _views_ para apoio às análises de dados previstas.

In [0]:
CREATE TABLE IF NOT EXISTS mvp_puc_engenhariadedados.gold.dim_tempo (
    sk_data         INTEGER NOT NULL PRIMARY KEY,
    data_completa   DATE NOT NULL,
    dia             INTEGER NOT NULL,
    mes             INTEGER NOT NULL,
    nome_mes        STRING NOT NULL,
    trimestre       INTEGER NOT NULL,
    semestre        INTEGER NOT NULL,
    ano             INTEGER NOT NULL
);

In [0]:
CREATE TABLE IF NOT EXISTS mvp_puc_engenhariadedados.gold.dim_produto (
    sk_produto  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    co_catmat   BIGINT NOT NULL,
    ds_item     STRING,
    co_pdm      BIGINT,
    no_pdm      STRING,
    co_grupo    BIGINT,
    no_grupo    STRING,
    co_classe   BIGINT,
    no_classe   STRING,

    CONSTRAINT uk_dim_produto_catmat
        UNIQUE (co_catmat)
);

In [0]:
CREATE TABLE IF NOT EXISTS mvp_puc_engenhariadedados.gold.dim_instituicao (
    sk_instituicao      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    cnpj_instituicao    STRING,
    no_instituicao      STRING,
    no_municipio        STRING,
    sg_uf               STRING,
    ds_esfera           STRING,

    CONSTRAINT uk_dim_instituicao_cnpj
        UNIQUE (cnpj_instituicao)
);

In [0]:
CREATE TABLE IF NOT EXISTS mvp_puc_engenhariadedados.gold.dim_fornecedor (
    sk_fornecedor       BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    cnpj_fornecedor     STRING,
    no_fornecedor       STRING,

    CONSTRAINT uk_dim_fornecedor_cnpj
        UNIQUE (cnpj_fornecedor)
);

In [0]:
CREATE TABLE IF NOT EXISTS mvp_puc_engenhariadedados.gold.dim_fabricante (
    sk_fabricante       BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    cnpj_fabricante     STRING,
    no_fabricante       STRING,

    CONSTRAINT uk_dim_fabricante_cnpj
        UNIQUE (cnpj_fabricante)
);

In [0]:
CREATE TABLE IF NOT EXISTS mvp_puc_engenhariadedados.gold.fato_compra (
    sk_compra              BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Identificador original do BPS
    co_seq_bps              BIGINT NOT NULL,

    -- Chaves estrangeiras
    sk_produto              BIGINT NOT NULL,
    sk_instituicao          BIGINT,
    sk_fornecedor           BIGINT,
    sk_fabricante           BIGINT,
    sk_data_compra          INTEGER NOT NULL,

    -- Controle temporal e de carga
    ano_compra              INTEGER NOT NULL,
    ano_arquivo             INTEGER NOT NULL,
    dt_insercao             DATE,

    -- Características da compra
    tp_compra               STRING,
    modalidade              STRING,
    validade_compra         INTEGER,
    nu_processo_compra      STRING,
    nu_ata                  STRING,

    -- Características do item no registro da compra
    fg_generico             STRING,
    sg_unidade_medida       STRING,
    un_medida_capacidade    STRING,
    un_fornecimento         STRING,
    vl_capacidade           DECIMAL(18,6),
    registro_anvisa         STRING,

    -- Quantidades e valores
    qt_medicamento          DECIMAL(18,6),
    vl_preco_unitario       DECIMAL(18,6),
    vl_preco_total          DECIMAL(18,6),

    -- Observação do registro
    ds_observacao           STRING,

    -- Restrições
    CONSTRAINT uk_fato_compra_bps
        UNIQUE (co_seq_bps),

    CONSTRAINT fk_fato_produto
        FOREIGN KEY (sk_produto)
        REFERENCES mvp_puc_engenhariadedados.gold.dim_produto (sk_produto),

    CONSTRAINT fk_fato_instituicao
        FOREIGN KEY (sk_instituicao)
        REFERENCES mvp_puc_engenhariadedados.gold.dim_instituicao (sk_instituicao),

    CONSTRAINT fk_fato_fornecedor
        FOREIGN KEY (sk_fornecedor)
        REFERENCES mvp_puc_engenhariadedados.gold.dim_fornecedor (sk_fornecedor),

    CONSTRAINT fk_fato_fabricante
        FOREIGN KEY (sk_fabricante)
        REFERENCES mvp_puc_engenhariadedados.gold.dim_fabricante (sk_fabricante),

    CONSTRAINT fk_fato_tempo
        FOREIGN KEY (sk_data_compra)
        REFERENCES mvp_puc_engenhariadedados.gold.dim_tempo (sk_data)
);

In [0]:
INSERT INTO mvp_puc_engenhariadedados.gold.dim_tempo (
    sk_data, data_completa, dia, mes, nome_mes, trimestre, semestre, ano
)
SELECT
    -- Chave derivada da própria data no formato AAAAMMDD
    -- Exemplo: 2026-01-01 → 20260101
    CAST(date_format(data_completa, 'yyyyMMdd') AS INTEGER) AS sk_data,

    data_completa,
    day(data_completa) AS dia,
    month(data_completa) AS mes,

    CASE month(data_completa)
        WHEN 1 THEN 'Janeiro'
        WHEN 2 THEN 'Fevereiro'
        WHEN 3 THEN 'Março'
        WHEN 4 THEN 'Abril'
        WHEN 5 THEN 'Maio'
        WHEN 6 THEN 'Junho'
        WHEN 7 THEN 'Julho'
        WHEN 8 THEN 'Agosto'
        WHEN 9 THEN 'Setembro'
        WHEN 10 THEN 'Outubro'
        WHEN 11 THEN 'Novembro'
        WHEN 12 THEN 'Dezembro'
    END AS nome_mes,

    quarter(data_completa) AS trimestre,

    CASE
        WHEN month(data_completa) <= 6 THEN 1
        ELSE 2
    END AS semestre,

    year(data_completa) AS ano

FROM (
    SELECT DISTINCT dt_compra AS data_completa
    FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
    WHERE dt_compra IS NOT NULL
) t;

In [0]:
INSERT INTO mvp_puc_engenhariadedados.gold.dim_produto (
    co_catmat, ds_item, co_pdm, no_pdm, co_grupo, no_grupo, co_classe, no_classe
)
SELECT
    co_catmat, ds_item, co_pdm, no_pdm, co_grupo, no_grupo, co_classe, no_classe
FROM (
    SELECT
        co_catmat, ds_item, co_pdm, no_pdm, co_grupo, no_grupo, co_classe, no_classe,

        ROW_NUMBER() OVER (
            PARTITION BY co_catmat
            ORDER BY
                dt_compra DESC,
                dt_insercao DESC,
                co_seq_bps DESC
        ) AS rn

    FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
    WHERE co_catmat IS NOT NULL
) t
WHERE rn = 1;

In [0]:
INSERT INTO mvp_puc_engenhariadedados.gold.dim_instituicao (
    cnpj_instituicao, no_instituicao, no_municipio, sg_uf, ds_esfera
)
SELECT
    cnpj_instituicao, no_instituicao, no_municipio, sg_uf, ds_esfera
FROM (
    SELECT
        cnpj_instituicao, no_instituicao, no_municipio, sg_uf, ds_esfera,

        ROW_NUMBER() OVER (
            PARTITION BY cnpj_instituicao
            ORDER BY
                dt_compra DESC,
                dt_insercao DESC,
                co_seq_bps DESC
        ) AS rn

    FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
    WHERE cnpj_instituicao IS NOT NULL
) t
WHERE rn = 1;

In [0]:
INSERT INTO mvp_puc_engenhariadedados.gold.dim_fornecedor (
    cnpj_fornecedor, no_fornecedor
)
SELECT
    cnpj_fornecedor, no_fornecedor
FROM (
    SELECT
        cnpj_fornecedor, no_fornecedor,

        ROW_NUMBER() OVER (
            PARTITION BY cnpj_fornecedor
            ORDER BY
                dt_compra DESC,
                dt_insercao DESC,
                co_seq_bps DESC
        ) AS rn

    FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
    WHERE cnpj_fornecedor IS NOT NULL
) t
WHERE rn = 1;

In [0]:
INSERT INTO mvp_puc_engenhariadedados.gold.dim_fabricante (
    cnpj_fabricante, no_fabricante
)
SELECT
    cnpj_fabricante, no_fabricante
FROM (
    SELECT
        cnpj_fabricante, no_fabricante,

        ROW_NUMBER() OVER (
            PARTITION BY cnpj_fabricante
            ORDER BY
                dt_compra DESC,
                dt_insercao DESC,
                co_seq_bps DESC
        ) AS rn

    FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
    WHERE cnpj_fabricante IS NOT NULL
) t
WHERE rn = 1;

In [0]:
INSERT INTO mvp_puc_engenhariadedados.gold.fato_compra (
    co_seq_bps,

    sk_produto,
    sk_instituicao,
    sk_fornecedor,
    sk_fabricante,
    sk_data_compra,

    dt_insercao,
    ano_compra,
    ano_arquivo,

    tp_compra,
    modalidade,
    validade_compra,

    nu_processo_compra,
    nu_ata,

    fg_generico,

    sg_unidade_medida,
    un_medida_capacidade,
    un_fornecimento,
    vl_capacidade,

    registro_anvisa,

    qt_medicamento,
    vl_preco_unitario,
    vl_preco_total,

    ds_observacao
)
SELECT
    s.co_seq_bps,

    p.sk_produto,
    i.sk_instituicao,
    f.sk_fornecedor,
    fa.sk_fabricante,
    t.sk_data,

    s.dt_insercao,
    s.ano_compra,
    s.ano_arquivo,

    s.tp_compra,
    s.modalidade,
    s.validade_compra,

    s.nu_processo_compra,
    s.nu_ata,

    s.fg_generico,

    s.sg_unidade_medida,
    s.un_medida_capacidade,
    s.un_fornecimento,
    s.vl_capacidade,

    s.registro_anvisa,

    s.qt_medicamento,
    s.vl_preco_unitario,
    s.vl_preco_total,

    s.ds_observacao

FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026 s

LEFT JOIN mvp_puc_engenhariadedados.gold.dim_produto p
    ON s.co_catmat = p.co_catmat

LEFT JOIN mvp_puc_engenhariadedados.gold.dim_instituicao i
    ON s.cnpj_instituicao = i.cnpj_instituicao

LEFT JOIN mvp_puc_engenhariadedados.gold.dim_fornecedor f
    ON s.cnpj_fornecedor = f.cnpj_fornecedor

LEFT JOIN mvp_puc_engenhariadedados.gold.dim_fabricante fa
    ON s.cnpj_fabricante = fa.cnpj_fabricante

LEFT JOIN mvp_puc_engenhariadedados.gold.dim_tempo t
    ON t.data_completa = s.dt_compra;

In [0]:
SELECT
    COUNT(*) AS qtd_registros,
    SUM(CASE WHEN sk_produto IS NULL THEN 1 ELSE 0 END) AS sem_produto,
    SUM(CASE WHEN sk_instituicao IS NULL THEN 1 ELSE 0 END) AS sem_instituicao,
    SUM(CASE WHEN sk_fornecedor IS NULL THEN 1 ELSE 0 END) AS sem_fornecedor,
    SUM(CASE WHEN sk_fabricante IS NULL THEN 1 ELSE 0 END) AS sem_fabricante,
    SUM(CASE WHEN sk_data_compra IS NULL THEN 1 ELSE 0 END) AS sem_data
FROM mvp_puc_engenhariadedados.gold.fato_compra;

In [0]:
CREATE OR REPLACE VIEW mvp_puc_engenhariadedados.gold.vw_registros_elegiveis AS

WITH catmats_elegiveis AS (
    SELECT
        p.co_catmat,
        COUNT(*) AS qtd_compras
    FROM mvp_puc_engenhariadedados.gold.fato_compra f
    INNER JOIN mvp_puc_engenhariadedados.gold.dim_produto p
        ON f.sk_produto = p.sk_produto
    WHERE f.vl_preco_unitario IS NOT NULL
      AND f.vl_preco_unitario > 0
    GROUP BY p.co_catmat
    HAVING COUNT(*) >= 5
)

SELECT
    f.co_seq_bps,
    f.ano_compra,
    t.data_completa,
    p.co_catmat,
    p.ds_item,
    f.vl_preco_unitario,
    f.qt_medicamento,
    f.vl_preco_total,
    f.tp_compra,
    f.modalidade,
    i.sg_uf,
    i.no_municipio,
    i.cnpj_instituicao,
    i.no_instituicao,
    fo.cnpj_fornecedor,
    fo.no_fornecedor
FROM mvp_puc_engenhariadedados.gold.fato_compra f
INNER JOIN mvp_puc_engenhariadedados.gold.dim_produto p
    ON f.sk_produto = p.sk_produto
INNER JOIN catmats_elegiveis ce
    ON p.co_catmat = ce.co_catmat
LEFT JOIN mvp_puc_engenhariadedados.gold.dim_tempo t
    ON f.sk_data_compra = t.sk_data
LEFT JOIN mvp_puc_engenhariadedados.gold.dim_instituicao i
    ON f.sk_instituicao = i.sk_instituicao
LEFT JOIN mvp_puc_engenhariadedados.gold.dim_fornecedor fo
    ON f.sk_fornecedor = fo.sk_fornecedor
WHERE f.vl_preco_unitario IS NOT NULL
  AND f.vl_preco_unitario > 0;

In [0]:
CREATE OR REPLACE VIEW mvp_puc_engenhariadedados.gold.vw_potenciais_outliers AS

WITH estatisticas_catmat AS (
    SELECT
        co_catmat,
        COUNT(*) AS qtd_compras,
        PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000) AS q1,
        PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000) AS q3
    FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis
    GROUP BY co_catmat
),
limites_catmat AS (
    SELECT
        co_catmat,
        qtd_compras,
        q1,
        q3,
        q3 - q1 AS iqr,
        q1 - 1.5 * (q3 - q1) AS limite_inferior,
        q3 + 1.5 * (q3 - q1) AS limite_superior
    FROM estatisticas_catmat
)

SELECT
    e.co_seq_bps,
    e.ano_compra,
    e.data_completa,
    e.co_catmat,
    e.ds_item,
    ROUND(e.vl_preco_unitario, 2) AS vl_preco_unitario,
    ROUND(l.q1, 2) AS q1,
    ROUND(l.q3, 2) AS q3,
    ROUND(l.iqr, 2) AS iqr,
    ROUND(l.limite_inferior, 2) AS limite_inferior,
    ROUND(l.limite_superior, 2) AS limite_superior,
    CASE
        WHEN e.vl_preco_unitario < l.limite_inferior
            THEN 'Abaixo do limite inferior'
        WHEN e.vl_preco_unitario > l.limite_superior
            THEN 'Acima do limite superior'
    END AS tipo_atipicidade,

    e.qt_medicamento,
    e.tp_compra,
    e.modalidade,
    e.sg_uf,
    e.no_municipio,
    e.cnpj_instituicao,
    e.no_instituicao,
    e.cnpj_fornecedor,
    e.no_fornecedor
FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e

INNER JOIN limites_catmat l
    ON e.co_catmat = l.co_catmat

WHERE e.vl_preco_unitario < l.limite_inferior
    OR e.vl_preco_unitario > l.limite_superior

ORDER BY
    e.co_catmat,
    e.vl_preco_unitario DESC;